# 17. View Classifier (5 kelas): frontal / lateral kanan / lateral kiri / maxillary / mandibular

Gerbang klasifikasi sudut foto sebelum DHC, AC, dan pembentukan 3D model. Data diambil dari
`data/view5/{train,val,test}/{kelas}/` yang sudah dibangun di **notebook 16** berdasarkan
ID pasien + split di `Grade AC by Team.xlsx`.

Catatan: notebook ini baru mencakup **5 kelas view valid**. Kelas ke-6 ("lainnya" / bukan foto
gigi yang valid) belum termasuk — perlu data negatif terpisah, ditambahkan belakangan sebagai
iterasi lanjutan (retrain jadi 6 kelas).

In [ ]:
import os, time, copy
import random
from glob import glob
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.expanduser('~/IOTN-AC')
IMG_DIR = os.path.join(PROJECT_ROOT, 'data', 'view5')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models'); os.makedirs(MODELS_DIR, exist_ok=True)
SPLITS = ['train', 'val', 'test']

# urutan kelas TETAP (index label mengikuti urutan ini di semua sel berikutnya)
VIEW_CLASSES = ['frontal', 'lateral_kanan', 'lateral_kiri', 'maxillary', 'mandibular']

PERANGKAT = ('cuda' if torch.cuda.is_available()
             else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()
                   else 'cpu'))
print('Perangkat:', PERANGKAT, '| IMG_DIR:', IMG_DIR)
assert os.path.isdir(IMG_DIR), f'Folder tidak ditemukan: {IMG_DIR} — jalankan notebook 16 dulu.'

In [ ]:
def muat_view5():
    """Scan data/view5/{split}/{kelas}/*.JPG -> {split: {'path': [...], 'y': [...]}}."""
    data = {s: {'path': [], 'y': []} for s in SPLITS}
    for s in SPLITS:
        for i_kls, kls in enumerate(VIEW_CLASSES):
            for p in sorted(glob(os.path.join(IMG_DIR, s, kls, '*.[Jj][Pp][Gg]'))):
                data[s]['path'].append(p); data[s]['y'].append(i_kls)
    return data

DATA = muat_view5()
print(f'{"split":6s} {"total":>6s}  ' + '  '.join(f'{k:>14s}' for k in VIEW_CLASSES))
for s in SPLITS:
    n = len(DATA[s]['path'])
    cnt = Counter(DATA[s]['y'])
    baris = '  '.join(f'{cnt.get(i,0):>14d}' for i in range(len(VIEW_CLASSES)))
    print(f'{s:6s} {n:>6d}  {baris}')

assert all(len(DATA[s]["path"]) for s in SPLITS), 'Ada split yang kosong — cek hasil notebook 16.'

In [ ]:
# --- augmentasi retractor-invariance: sebagian data training di-crop/ditutup
# di area pinggir supaya model TIDAK menjadikan 'ada retractor kelihatan' sebagai
# syarat foto valid -- seluruh dataset yang ada memang selalu pakai retractor,
# jadi tanpa ini model gampang menolak foto asli tanpa retractor (shortcut learning).
class ZoomKeTengah:
    """Dengan probabilitas p, crop lebih dekat ke tengah foto sebelum di-resize --
    mensimulasikan framing tanpa retractor kelihatan (retractor biasanya nongol di
    pinggir kiri-kanan foto). Sisanya (1-p) foto dipakai utuh seperti biasa, supaya
    model tetap kenal foto asli (dengan retractor)."""
    def __init__(self, p=0.5, skala=(0.55, 0.85)):
        self.p = p; self.skala = skala
    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        s = random.uniform(*self.skala)
        cw, ch = max(1, int(w * s)), max(1, int(h * s))
        max_dx = max(0, (w - cw) // 2); max_dy = max(0, (h - ch) // 2)
        x0 = w // 2 - cw // 2 + (random.randint(-max_dx, max_dx) if max_dx else 0)
        y0 = h // 2 - ch // 2 + (random.randint(-max_dy, max_dy) if max_dy else 0)
        x0 = max(0, min(x0, w - cw)); y0 = max(0, min(y0, h - ch))
        return img.crop((x0, y0, x0 + cw, y0 + ch))

class HapusPinggir:
    """Dengan probabilitas p, tutup sebagian pinggir kiri dan/atau kanan foto dengan
    warna acak -- mensimulasikan retractor yang tidak kelihatan/terpotong di sisi
    tertentu, beda dari ZoomKeTengah yang mengubah keseluruhan framing."""
    def __init__(self, p=0.35, lebar=(0.08, 0.18)):
        self.p = p; self.lebar = lebar
    def __call__(self, img):
        if random.random() > self.p:
            return img
        img = img.copy()
        w, h = img.size
        draw = ImageDraw.Draw(img)
        warna = tuple(random.randint(0, 255) for _ in range(3))
        sisi = random.choice(['kiri', 'kanan', 'keduanya'])
        lw = int(w * random.uniform(*self.lebar))
        if sisi in ('kiri', 'keduanya'):
            draw.rectangle([0, 0, lw, h], fill=warna)
        if sisi in ('kanan', 'keduanya'):
            draw.rectangle([w - lw, 0, w, h], fill=warna)
        return img

# --- augmentasi: TANPA horizontal flip (flip menukar makna kanan <-> kiri) ---
RATA, SIMPANG = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([
    ZoomKeTengah(p=0.5, skala=(0.55, 0.85)),
    HapusPinggir(p=0.35, lebar=(0.08, 0.18)),
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.92, 1.08)),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.03),
    transforms.ToTensor(), transforms.Normalize(RATA, SIMPANG),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.08))])
tf_eval = transforms.Compose([transforms.Resize((224, 224)),
                              transforms.ToTensor(), transforms.Normalize(RATA, SIMPANG)])

class DataView(Dataset):
    def __init__(self, paths, labels, tf): self.paths = paths; self.labels = labels; self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = self.tf(Image.open(self.paths[i]).convert('RGB'))
        return img, torch.tensor(self.labels[i], dtype=torch.long)

def loader(split, tf, shuffle):
    return DataLoader(DataView(DATA[split]['path'], DATA[split]['y'], tf),
                       batch_size=16, shuffle=shuffle, num_workers=0)

In [ ]:
def buat_resnet_cls(num_classes=len(VIEW_CLASSES), dropout=0.3):
    m = models.resnet18(weights='DEFAULT')
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    return m

def acak(s=42):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s)

@torch.no_grad()
def prediksi(model, dl):
    model.eval(); logit_all, y_all = [], []
    for x, y in dl:
        logit_all.append(model(x.to(PERANGKAT)).cpu().numpy()); y_all.append(y.numpy())
    logit = np.concatenate(logit_all); y = np.concatenate(y_all)
    pred = logit.argmax(1)
    return pred, logit, y

In [ ]:
def latih_cls(epoch=20, lr=1e-4, wd=5e-2, seed=42, pakai_class_weight=True):
    acak(seed)
    model = buat_resnet_cls().to(PERANGKAT)
    dl_tr = loader('train', tf_train, True)
    dl_va = loader('val', tf_eval, False)

    if pakai_class_weight:
        cnt = Counter(DATA['train']['y'])
        w = torch.tensor([1.0 / cnt.get(i, 1) for i in range(len(VIEW_CLASSES))], dtype=torch.float32)
        w = (w / w.sum() * len(VIEW_CLASSES)).to(PERANGKAT)
        print('Class weight (imbang kelas minoritas):', {VIEW_CLASSES[i]: round(float(w[i]), 3) for i in range(len(VIEW_CLASSES))})
    else:
        w = None
    crit = nn.CrossEntropyLoss(weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epoch)
    ckpt_dir = os.path.join(MODELS_DIR, 'checkpoints', 'resnet18_view5')
    os.makedirs(ckpt_dir, exist_ok=True)

    terbaik, bobot = -1.0, None
    riwayat = {'train_loss': [], 'val_acc': []}
    for ep in range(epoch):
        model.train(); total = 0.0; n = 0
        for x, y in dl_tr:
            x = x.to(PERANGKAT); y = y.to(PERANGKAT)
            loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(x); n += len(x)
        sched.step()
        l_tr = total / max(n, 1)
        pv, _, yv = prediksi(model, dl_va)
        acc = float((pv == yv).mean())
        riwayat['train_loss'].append(l_tr); riwayat['val_acc'].append(acc)

        ckpt = dict(epoch=ep + 1, val_acc=acc, model=model.state_dict(), optim=opt.state_dict())
        torch.save(ckpt, os.path.join(ckpt_dir, 'last.pt'))
        tanda = ''
        if acc > terbaik:
            terbaik = acc; bobot = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(ckpt, os.path.join(ckpt_dir, 'best.pt')); tanda = '  <- terbaik (checkpoint)'
        print(f'  epoch {ep + 1:2d}/{epoch}  train {l_tr:.4f}  val_acc {acc:.3%}{tanda}')
    model.load_state_dict(bobot)
    print(f'  checkpoint: {os.path.relpath(ckpt_dir, PROJECT_ROOT)}/ (best.pt, last.pt)')
    return model, terbaik, riwayat

model, val_acc_terbaik, RIWAYAT = latih_cls(epoch=20)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ep = range(1, len(RIWAYAT['train_loss']) + 1)
ax[0].plot(ep, RIWAYAT['train_loss'], 'o-', color='#ED7D31'); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, RIWAYAT['val_acc'], 'o-', color='#4472C4'); ax[1].set_title('Val accuracy'); ax[1].set_xlabel('epoch')
ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()
print(f'Val accuracy terbaik: {val_acc_terbaik:.3%}')

## Evaluasi di test set

Confusion matrix + precision/recall/F1 per kelas. Perhatikan khusus pasangan yang rawan
ketuker: **lateral_kanan vs lateral_kiri**, dan **maxillary vs mandibular**.

In [ ]:
dl_test = loader('test', tf_eval, False)
pred_test, logit_test, y_test = prediksi(model, dl_test)
acc_test = float((pred_test == y_test).mean())
print(f'Test accuracy: {acc_test:.3%}  (n={len(y_test)})')

K = len(VIEW_CLASSES)
CM = np.zeros((K, K), int)
for t, p in zip(y_test, pred_test): CM[t, p] += 1

fig, ax = plt.subplots(figsize=(6.5, 5.8))
im = ax.imshow(CM, cmap='Blues')
ax.set_xticks(range(K)); ax.set_xticklabels(VIEW_CLASSES, rotation=35, ha='right')
ax.set_yticks(range(K)); ax.set_yticklabels(VIEW_CLASSES)
ax.set_xlabel('prediksi'); ax.set_ylabel('label sebenarnya'); ax.set_title('Confusion matrix — test set')
for t in range(K):
    for p in range(K):
        if CM[t, p]:
            ax.text(p, t, CM[t, p], ha='center', va='center', fontsize=9,
                    color='white' if CM[t, p] > CM.max()/2 else 'black')
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
_out = os.path.join(PROJECT_ROOT, 'outputs', 'confusion_matrix_view5.png')
os.makedirs(os.path.dirname(_out), exist_ok=True); fig.savefig(_out, dpi=130, bbox_inches='tight'); plt.show()
print('disimpan ke', os.path.relpath(_out, PROJECT_ROOT))

In [ ]:
print(f'{"kelas":>14s} {"n":>5s} {"precision":>10s} {"recall":>8s} {"F1":>6s}')
print('-' * 48)
for i, kls in enumerate(VIEW_CLASSES):
    tp = CM[i, i]; fp = CM[:, i].sum() - tp; fn = CM[i, :].sum() - tp
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    print(f'{kls:>14s} {CM[i,:].sum():>5d} {prec:>9.1%} {rec:>7.1%} {f1:>6.1%}')
print('-' * 48)
print(f'{"akurasi total":>14s} {len(y_test):>5d} {"":>9s} {"":>7s} {acc_test:>6.1%}')

## Simpan model final

`best.pt` (dari checkpoint) sudah tersimpan otomatis di `models/checkpoints/resnet18_view5/`.
Sel di bawah menyalin bobot itu ke `models/view5_classifier.pt` sebagai nama final yang
dipakai untuk ekspor Core ML nanti (mengikuti pola `ac_resnet18_*.pt` di notebook AC).

In [ ]:
_final_path = os.path.join(MODELS_DIR, 'view5_classifier.pt')
torch.save(model.state_dict(), _final_path)
print('Model view classifier tersimpan ->', os.path.relpath(_final_path, PROJECT_ROOT))
print('\nUrutan kelas (index label) — WAJIB dipakai sama persis saat inferensi/ekspor:')
for i, k in enumerate(VIEW_CLASSES): print(f'  {i}: {k}')

## Uji satu foto

Ganti `GAMBAR_UJI` ke path foto yang mau dicek, lalu jalankan sel di bawah. Model dimuat ulang
dari `models/view5_classifier.pt` (bukan pakai variabel `model` di memori) supaya sel ini bisa
dijalankan sendiri kapan saja tanpa harus training ulang dulu.

In [ ]:
# ============ Uji satu foto: input gambar -> prediksi view ============
GAMBAR_UJI = '/ganti/dengan/path/foto.jpg'   # <-- ganti path ini ke foto yang mau dicek

assert os.path.exists(GAMBAR_UJI), f'File tidak ditemukan: {GAMBAR_UJI}'

_model_uji = buat_resnet_cls().to(PERANGKAT)
_ckpt_path = os.path.join(MODELS_DIR, 'view5_classifier.pt')
assert os.path.exists(_ckpt_path), f'Model belum ada: {_ckpt_path} — jalankan training di atas dulu.'
_model_uji.load_state_dict(torch.load(_ckpt_path, map_location=PERANGKAT))
_model_uji.eval()

img = Image.open(GAMBAR_UJI).convert('RGB')
with torch.no_grad():
    logit = _model_uji(tf_eval(img).unsqueeze(0).to(PERANGKAT))
    prob = torch.softmax(logit, 1).cpu().numpy()[0]

pred_i = int(prob.argmax())
print(f'Prediksi: {VIEW_CLASSES[pred_i]}  (confidence {prob[pred_i]:.1%})\n')
print('Probabilitas tiap kelas:')
for i, kls in enumerate(VIEW_CLASSES):
    tanda = '  <-' if i == pred_i else ''
    print(f'  {kls:>14s}: {prob[i]:>6.1%}{tanda}')

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title(f'Prediksi: {VIEW_CLASSES[pred_i]}  ({prob[pred_i]:.1%})')
plt.show()

## Langkah lanjutan (belum di notebook ini)

- **Kelas ke-6 "lainnya"** (bukan foto gigi valid / near-miss) — retrain jadi 6-kelas setelah
  data negatif terkumpul.
- **Ekspor Core ML** — bungkus dengan normalisasi tertanam + `ct.convert`, mengikuti pola
  persis seperti `ACGraderCoreML` di notebook 10, lalu jadi gerbang sebelum DHC/AC/3D model di app.